In [2]:
import sys
import os
model_path = os.path.join("/home/gridsan/manderson/ovdsat/Long-CLIP/model")
sys.path.append(os.path.abspath(model_path))
import longclip

import torch
from PIL import Image

In [3]:
ckpt_dir = '/home/gridsan/manderson/ovdsat/Long-CLIP/checkpoints/005-07--05_28_20_longclip.pt'

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = longclip.load(ckpt_dir, device=device)

text = longclip.tokenize(["A man is crossing the street with a red car parked nearby.", "A man is driving a car in an urban scene."]).to(device)
image = preprocess(Image.open("Long-CLIP/img/demo.png")).unsqueeze(0).to(device)

with torch.no_grad():
    image_features = model.encode_image(image)
    print(image_features.shape)
    text_features = model.encode_text(text)
    print(text_features.shape)
    
    logits_per_image = image_features @ text_features.T
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()

print("Label probs:", probs) 

torch.Size([1, 768])
torch.Size([2, 768])
Label probs: [[0.998   0.00181]]


In [4]:
ckpt = torch.load(ckpt_dir)

In [5]:
ckpt

OrderedDict([('positional_embedding',
              tensor([[ 0.0016,  0.0020,  0.0002,  ..., -0.0013,  0.0008,  0.0015],
                      [ 0.0042,  0.0029,  0.0002,  ...,  0.0010,  0.0015, -0.0012],
                      [ 0.0018,  0.0007, -0.0012,  ..., -0.0029, -0.0009,  0.0026],
                      ...,
                      [ 0.0365,  0.0333,  0.0380,  ...,  0.0206,  0.0130, -0.0401],
                      [ 0.0401,  0.0385,  0.0471,  ...,  0.0252,  0.0158, -0.0493],
                      [ 0.0436,  0.0437,  0.0563,  ...,  0.0298,  0.0185, -0.0584]],
                     device='cuda:0')),
             ('text_projection',
              tensor([[-0.0109,  0.0099, -0.0036,  ..., -0.0010,  0.0111, -0.0037],
                      [-0.0049, -0.0046,  0.0057,  ...,  0.0239,  0.0173, -0.0073],
                      [ 0.0033,  0.0097, -0.0157,  ...,  0.0068, -0.0115, -0.0097],
                      ...,
                      [-0.0107,  0.0009,  0.0023,  ..., -0.0173, -0.0097, -0.0

In [17]:
image_features[:, :10]

tensor([[ 0.8633,  0.5366, -1.0107,  1.5791, -0.1498, -0.0283,  0.4417, -0.4915,
          0.2042, -0.3591]], device='cuda:0', dtype=torch.float16)

In [6]:
image = image.half()
patch_tokens = model.visual(image, return_tokens=True)

In [12]:
model_vision = model.visual

In [15]:
patch_tokens = model_vision(image, return_tokens=True)
print(patch_tokens.shape)

torch.Size([1, 256, 1024])


In [10]:
print(patch_tokens.shape)

torch.Size([1, 256, 1024])


In [14]:
image = image.half()
img_feat = model.visual(image)

In [21]:
img_feat.shape

torch.Size([1, 768])

In [18]:
img_feat[:, :10]

tensor([[ 0.8643,  0.5366, -1.0107,  1.5791, -0.1512, -0.0295,  0.4417, -0.4912,
          0.2036, -0.3594]], device='cuda:0', dtype=torch.float16,
       grad_fn=<SliceBackward0>)

In [10]:
model

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
          )
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
